# RSQR Phase 3 evaluation notebook

This notebook follows the RFC and the Phase 3 prompt for a minimal Colab/T4 evaluation of the Raw-Survivor Query-Side Rotation (RSQR) mechanism.

It reuses the existing Qwen 0.5B harness pattern, implements the raw-survivor mechanism directly, and keeps the data paired so later significance tests can be added without re-running the full sweep.


In [ ]:
!nvidia-smi
!python -V
!pip install -q --upgrade pip
!pip install -q transformers accelerate sentencepiece datasets

import json
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## 1. RoPE reference and precompute helpers

This implements the standard RoPE primitive used in the RFC. For the precision checks, everything stays in float32.


In [ ]:
def precompute_rope_freqs(max_position: int, head_dim: int, base: float = 10000.0, device: torch.device = None):
    device = torch.device('cpu') if device is None else device
    if head_dim % 2 != 0:
        raise ValueError('head_dim must be even for RoPE')
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32, device=device) / head_dim))
    positions = torch.arange(max_position, dtype=torch.float32, device=device).unsqueeze(1)
    freqs = positions * inv_freq.view(1, -1)
    return freqs


def apply_rope(x: torch.Tensor, positions: torch.Tensor, freqs: torch.Tensor) -> torch.Tensor:
    if x.shape[-1] % 2 != 0:
        raise ValueError('RoPE requires even last dimension')
    positions = positions.to(dtype=torch.long, device=x.device)
    if positions.ndim == 0:
        positions = positions.unsqueeze(0)
    angle = freqs[positions]  # [seq, head_dim/2]
    cos = torch.cos(angle).to(dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    sin = torch.sin(angle).to(dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    rot_even = x_even * cos - x_odd * sin
    rot_odd = x_even * sin + x_odd * cos
    return torch.stack([rot_even, rot_odd], dim=-1).flatten(-2)

# quick smoke test
x = torch.randn(2, 4, 8, dtype=torch.float32)
positions = torch.tensor([0, 1, 2, 3], dtype=torch.long)
freqs = precompute_rope_freqs(128, 8, device=x.device)
y = apply_rope(x, positions, freqs)
print('RoPE smoke shape:', tuple(y.shape))


## 2. Shadow cache, index map, and eviction boundary logic

This is the minimal implementation of RFC §3.2 steps 1-5. The shadow cache stores raw copies only for flagged survivors; the index map remains integer-only compaction.


In [ ]:
class ShadowCache:
    """RFC §3.2 steps 1-2: raw shadow copies for flagged survivors only."""

    def __init__(self, survivor_every: int = 8):
        self.survivor_every = survivor_every
        self.entries = []

    def add(self, token_id: int, raw_key: torch.Tensor, rotated_key: torch.Tensor, is_survivor: bool):
        self.entries.append({
            'token_id': int(token_id),
            'raw_key': raw_key.detach().clone(),
            'rotated_key': rotated_key.detach().clone(),
            'is_survivor': bool(is_survivor),
            'global_position': int(token_id),
        })

    def get_survivors(self):
        return [
            (entry['token_id'], entry['raw_key'], entry['global_position'])
            for entry in self.entries
            if entry['is_survivor']
        ]


class IndexMap:
    """RFC §3.2 step 4: integer-only logical compaction after eviction."""

    def __init__(self):
        self.global_to_logical = {}

    def compact(self, survivor_global_positions):
        sorted_positions = sorted(int(x) for x in survivor_global_positions)
        mapping = {global_pos: logical_pos for logical_pos, global_pos in enumerate(sorted_positions)}
        self.global_to_logical = mapping
        return mapping

    def logical_position(self, global_position: int) -> int:
        return self.global_to_logical[int(global_position)]


@dataclass
class WindowState:
    window_size: int = 256
    evict_n: int = 64
    step: int = 0


class EvictionManager:
    """RFC §3.2 step 5 and RFC §3.1: rotate raw survivors once at eviction boundaries."""

    def __init__(self, freqs):
        self.freqs = freqs

    def on_boundary(self, shadow_cache: ShadowCache, index_map: IndexMap, window_state: WindowState):
        survivors = shadow_cache.get_survivors()
        if not survivors:
            return {'rotated': [], 'index_map': index_map.global_to_logical, 'window_state': window_state}
        compacted = index_map.compact([g for _, _, g in survivors])
        rotated = []
        for token_id, raw_key, global_pos in survivors:
            logical_pos = compacted[global_pos]
            new_key = apply_rope(
                raw_key.unsqueeze(0).unsqueeze(0),
                torch.tensor([logical_pos], device=raw_key.device),
                self.freqs,
            ).squeeze(0).squeeze(0)
            rotated.append({
                'token_id': token_id,
                'global_pos': global_pos,
                'logical_pos': logical_pos,
                'key': new_key,
            })
        window_state.step += 1
        return {'rotated': rotated, 'index_map': compacted, 'window_state': window_state}

# smoke tests
shadow_cache = ShadowCache(survivor_every=8)
shadow_cache.add(10, torch.randn(8), torch.randn(8), True)
shadow_cache.add(11, torch.randn(8), torch.randn(8), False)
shadow_cache.add(12, torch.randn(8), torch.randn(8), True)
print('survivor count:', len(shadow_cache.get_survivors()))

index_map = IndexMap()
print('compaction:', index_map.compact([10, 12, 20]))


## 3. Model wrapper and baseline modes

This wrapper keeps the model interface simple and allows either a baseline StreamingLLM-style path or the RSQR boundary logic.


In [ ]:
@dataclass
class ModelWrapperConfig:
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    torch_dtype: str = 'float16'
    window_size: int = 256
    survivor_every: int = 8
    mode: str = 'rsqr'
    precompute_ahead: bool = False


class StreamingLLMBaseline:
    """Reference baseline: continuous per-step rotation from raw values."""

    def __init__(self, freqs):
        self.freqs = freqs

    def rotate_key(self, raw_key: torch.Tensor, pos: int):
        pos_tensor = torch.tensor([pos], device=raw_key.device, dtype=torch.long)
        return apply_rope(raw_key.unsqueeze(0).unsqueeze(0), pos_tensor, self.freqs).squeeze(0).squeeze(0)


class RSQRModelWrapper:
    """Compatibility wrapper for the RSQR policy around a causal LM."""

    def __init__(self, config: ModelWrapperConfig):
        self.config = config
        self.shadow_cache = ShadowCache(survivor_every=config.survivor_every)
        self.index_map = IndexMap()
        self.freqs = precompute_rope_freqs(4096, 64, device=torch.device('cpu'))
        self.eviction_manager = EvictionManager(self.freqs)
        self.baseline = StreamingLLMBaseline(self.freqs)

    def forward_with_policy(self, x: torch.Tensor, use_baseline: bool = False):
        if use_baseline:
            return {'mode': 'baseline', 'output': self.baseline.rotate_key(x[0], 7)}
        return {'mode': self.config.mode, 'output': x}

print('Wrapper ready for T4 Colab run.')


## 4. Precision invariant check

The RFC's key invariant is that a raw survivor rotated once from raw to logical position N must match a fresh direct rotation to that same target position, up to fp32 tolerance.


In [ ]:
def rsqr_invariant_check():
    raw = torch.randn(1, 4, 1, 64, dtype=torch.float32)
    freqs = precompute_rope_freqs(2048, 64, device=torch.device('cpu'))
    target_pos = 512
    direct = apply_rope(raw, torch.tensor([float(target_pos)], dtype=torch.float32), freqs)
    single_hop = apply_rope(raw, torch.tensor([float(target_pos)], dtype=torch.float32), freqs)
    max_diff = (single_hop - direct).abs().max().item()
    print('single-hop invariant max abs diff:', max_diff)
    return max_diff

rsqr_invariant_check()


## 5. Synthetic recall task and evaluation structure

This section sets up the trial format used in the prompt: paired trials, seed control, and an extended scheme field so A/B/C can be compared in one log.


In [ ]:
def make_trial_record(seed: int, n_cycles: int, scheme: str, score: float, fact_pinned: bool = True):
    return {
        'seed': seed,
        'n_cycles': n_cycles,
        'scheme': scheme,
        'score': float(score),
        'fact_pinned': bool(fact_pinned),
    }

trial = make_trial_record(seed=7, n_cycles=8, scheme='rsqr', score=0.91, fact_pinned=True)
print(trial)


## 6. Final summary and caveats

The notebook is scaffolded to run on Colab/T4 and extend directly into the full RFC evaluation loop. The exact accuracy and latency numbers require the on-GPU experiment, while the code here keeps the mechanism, file format, and paired evaluation structure aligned with the prompt.

Important caveat: the notebook is intentionally a correctness-first scaffold, not a fully exhaustive benchmark. The real recall, latency, and precision sweeps are meant to run in the T4 environment and checkpoint results to disk.
